# Enclave Evaluation — Model Owner

| Actor | Email | Role |
|-------|-------|------|
| **Enclave** | `ENCLAVE_EMAIL` | Trusted execution environment |
| **Model owner** | (this notebook) | Owns the model weights (NanoLM) |
| **Benchmark owner** | (separate notebook) | Owns the demographic evaluation benchmark |
| **Researcher** | `RESEARCHER_EMAIL` | Writes the inference code, submits the job |

This notebook drives only the Model Owner steps; the Benchmark Owner and Researcher run their own notebooks in parallel.

In [ ]:
!uv pip install -Uq "git+https://github.com/OpenMined/syft-client.git@main#subdirectory=packages/syft-enclave"

## Setup

In [ ]:
import csv
import json
import os
import random
import tempfile
from pathlib import Path

os.environ["PRE_SYNC"] = "false"

from syft_enclaves import login_do, login_ds

In [ ]:
ENCLAVE_EMAIL    = "test.enclave@gmail.com"
RESEARCHER_EMAIL = "test.researcher@gmail.com"

print(f"  Enclave    : {ENCLAVE_EMAIL}\n  Researcher : {RESEARCHER_EMAIL}")

## Step 0 — Log in as Model Owner

In [ ]:
model_owner = login_do()
print(f"  Model owner : {model_owner.email}")

In [ ]:
# Optionally to clear state
# model_owner._manager.delete_syftbox()
# model_owner._manager.peer_manager.write_own_version()

## Step 1 — Peer with the Enclave

In [ ]:
model_owner.add_peer(ENCLAVE_EMAIL)
model_owner.sync()
print(f"  Model owner peered with enclave ({ENCLAVE_EMAIL})")

### Step 1.1 — Wait for the Researcher peer request, then approve

The Researcher notebook adds you as a peer. Re-run the cell below until you see their request appear, then approve.

In [ ]:
model_owner.peers

In [ ]:
model_owner.approve_peer_request(RESEARCHER_EMAIL, peer_must_exist=False)
model_owner.sync()
print("  Researcher peer approved")

## Step 2.1 — Dataset helpers

In [ ]:
# Model owner's NanoLM inference module variants and weights.
NANO_LM_CODE_PRIVATE = """
class NanoLMTokenizer:
    def encode(self, text: str) -> list[int]:
        return [ord(c) for c in text]

    def decode(self, ids: list[int]) -> str:
        return "".join(chr(i) for i in ids)


class NanoLM:
    def __init__(self):
        self.weights = None

    def init(self, weights):
        self.weights = weights

    def generate(self, prompt: str, max_new_tokens: int = 50) -> str:
        n = len(self.weights) if self.weights is not None else 0
        return f"[NanoLM({n}w) inference on: {prompt[:30]}...]"


tokenizer = NanoLMTokenizer()
model     = NanoLM()
"""

NANO_LM_CODE_MOCK = """
class NanoLM:
    def init(self, weights):
        self.weights = weights

    def generate(self, prompt, max_new_tokens=50):
        return f"[mock NanoLM({len(self.weights)}w) preview]"


model = NanoLM()
"""

MOCK_WEIGHTS    = [0.0, 0.0, 0.0]
PRIVATE_WEIGHTS = [0.11, 0.22, 0.33, 0.44, 0.55, 0.66, 0.77, 0.88, 0.99, 1.10]

In [ ]:
def create_model_dir(code: str, weights: list) -> Path:
    """Write nano_lm.py + weights.json to a temp directory."""
    tmp = Path(tempfile.mkdtemp()) / f"model-{random.randint(1, 1_000_000)}"
    tmp.mkdir(parents=True, exist_ok=True)
    (tmp / "nano_lm.py").write_text(code.strip())
    (tmp / "weights.json").write_text(json.dumps(weights))
    return tmp

## Step 2.2 — Upload the model weights

* **mock**: a public model card
* **private**: `weights.json` — only the enclave receives this

In [ ]:
model_owner.create_dataset(
    name="gemma_model",
    mock_path=create_model_dir(NANO_LM_CODE_MOCK,    MOCK_WEIGHTS),
    private_path=create_model_dir(NANO_LM_CODE_PRIVATE, PRIVATE_WEIGHTS),
    summary="NanoLM v1.0 — inference code (nano_lm.py) + weights (weights.json)",
    users=[RESEARCHER_EMAIL, ENCLAVE_EMAIL],
    upload_private=True,
    sync=False,
)
model_owner.share_private_dataset("gemma_model", ENCLAVE_EMAIL)
model_owner.sync()
print("  Model owner uploaded 'gemma_model'")

## Step 3 — Wait for the Researcher to submit the job, then approve

The Researcher submits `bias_eval_job` to the enclave. Re-sync until it appears here, inspect it, then approve.

In [ ]:
JOB_NAME = "bias_eval_job"
model_owner.sync()
model_owner_job = next(j for j in model_owner.jobs if j.name == JOB_NAME)
print(f"  Model owner sees '{JOB_NAME}'  status={model_owner_job.status}")

In [ ]:
model_owner.approve_job(model_owner_job)
print("  Model owner approved")

## Step 4 — Receive results

Results arrive here because the Researcher submitted with `share_results_with_do=True`.

In [ ]:
model_owner.sync()
model_owner_job = next(j for j in model_owner.jobs if j.name == JOB_NAME)
print(f"  Output files : {model_owner_job.output_paths}")
assert len(model_owner_job.output_paths) > 0